# Time series generation through Wiener filter based on whitening

This notebook demonstrates how to generate time series data using a Wiener filter based on whitening. The Wiener filter is a powerful tool for signal processing and can be used to create synthetic time series that mimic the statistical properties of real-world data.

在随机信号处理中的维纳滤波器是基于正交性原理设计的最优线性滤波器，旨在最小化滤波器输出与期望信号之间的均方误差。该滤波的设计依赖于输入信号的统计特性，特别是其自相关函数和功率谱密度。

假设我们的输入信号 $x(n)$是由我们期望信号$s(n)$和与期望信号不相关的噪声$v(n)$组成，即 $x(n) = s(n) + v(n)$。我们期望构造一个滤波器$h(n)$使得滤波器输出$y(n)$尽可能接近期望信号$s(n)$。即：
$$
y(n) = h(n) * x(n) = \sum_{m = 0}^{N - 1} h(m) \cdot x(n - m),  \quad (1)
$$  
其中$N$是滤波器的长度，$*$表示卷积运算。由于维纳滤波器的设计基于最小均方误差准则，我们需要最小化误差：
$$
E \left [ e ^ 2 \left (n \right ) \right ] = E \left [ (s(n) - y(n)) \right ] = E \left [ \left ( s(n) - (\sum_{m = 0}^{N - 1} h(m) \cdot x(n - m)) \right )^2 \right ]. \quad (2)
$$

为了便于得出矩阵形式，我们将(1)式改写为:
$$
y(n) = \sum_{i = 1}^{N} h(i) \cdot x(i),  \quad (3)
$$
此时均方误差可以进一步被简化为:
$$
E \left [ e ^ 2 \left (n \right ) \right ] = E \left [ \left ( s(n) - \sum_{i = 1}^{N} h(i) \cdot x(i) \right )^2 \right ]. \quad (4)
$$

为了求得$E \left [ e ^ 2 \left (n \right ) \right ]$的最小值，我们对$h(i)$求偏导数并令其为零。设$h = [h(1), h(2), ..., h(N)]^T$，则有：
$$
\frac{\partial}{\partial h(i)} E \left [ e ^ 2 \left (n \right ) \right ] = E \left [ \frac{\partial}{\partial h(i)} \left ( s(n) - \sum_{j = 1}^{N} h(j) \cdot x(j) \right )^2 \right ] = 0, \quad i = 1, 2, ..., N.
$$

对上式进行进一步的化简可得:
$$
E \left [ -2 \cdot x(i) \cdot \left ( s(n) - \sum_{j = 1}^{N} h(j) \cdot x(j) \right ) \right ] = 0, \quad i = 1, 2, ..., N.
$$
其中$s(n) - \sum_{j = 1}^{N} h(j) \cdot x(j)$为误差项$e(n)$，因此上式可以改写为:
$$
E \left [ x(i) \cdot e(n) \right ] = 0, \quad i = 1, 2, ..., N. \quad (5)
$$
即输入信号$x(i)$与误差$e(n)$之间的期望值为零，这表明输入信号与误差是正交的。（进一步来说，满足正交性原理与满足均方误差最小化的条件是一致的。）

假设信号$x_i$与信号$x_j$之间的相关函数为$r_{ij} = E[x_i x_j]$，信号$x_i$与期望信号$s(n)$之间的相关性为$p_i = E[x_i s(n)]$，则上述方程可以表示为:
$$
r_{x_j s} = \sum_{i = 1}^{N} h(i) \cdot r_{x_i x_j}. \quad j = 1, 2, ..., N. \quad (6)
$$

我们将(6)式中的下标复原为等式(1)的形式便有了:
$$
E \left \{ \left [ s(n) - \sum_{m = 0} ^ N h_{\mathrm{opt}} (m) x (n - m) \right ] \cdot x (n - k) \right \} = 0, \quad (7)
$$
其中$h_{\mathrm{opt}}$是维纳滤波器的最优系数。将其写为相关的形式可以得到:
$$
r_{s x} (k) = \sum_{m = 0} ^ N h_{\mathrm{opt}} (m) r_{x x} (k - m). \quad (8)
$$

式(8)便是基于正交性原理推导得到的维纳-霍夫方程。通过求解该方程，我们可以得到最优滤波器系数$h_{\mathrm{opt}}$，从而实现对输入信号的最佳线性估计。

为了表述方便和计算求解，我们通常将上述方程以矩阵形式表示。设$\left [h_{\mathrm{opt}} \right ] = [h_{\mathrm{opt}}(0), h_{\mathrm{opt}}(1), ..., h_{\mathrm{opt}}(N)]^T$，则可以将式(8)改写为:
$$
\begin{bmatrix}r_{x x} (0) & r_{x x} (1) & \cdots & r_{x x} (N) \\ r_{x x} (1) & r_{x x} (0) & \cdots & r_{x x} (N - 1) \\ \vdots & \vdots & \ddots & \vdots \\ r_{x x} (N) & r_{x x} (N - 1) & \cdots & r_{x x} (0) \end{bmatrix} \cdot \begin{bmatrix} h_{\mathrm{opt}} (0) \\ h_{\mathrm{opt}} (1) \\ \vdots \\ h_{\mathrm{opt}} (N) \end{bmatrix} = \begin{bmatrix} r_{s x} (0) \\ r_{s x} (1) \\ \vdots \\ r_{s x} (N) \end{bmatrix}. \quad (9)
$$

通过式(9)我们可以通过矩阵求逆和相乘的方式获得最优滤波器的参数:
$$
\left [ h_{\mathrm{opt}} \right ] = R_{x x} ^ {-1} \cdot r_{s x}. \quad (10)
$$

介绍完维纳滤波器的推导后，我们将要进一步介绍应该如何使用维纳滤波器来生成时间序列数据。具体来说，该过程与数据的白化处理密切相关。通过对输入信号进行白化处理，我们可以使得输入信号的自相关函数变为单位矩阵，从而简化维纳滤波器的设计和实现。

简单来说，对比白噪声$w(n)$，其自相关函数为$\delta(k)$，即只有在$k=0$时才为1，其他时候都为0。而我们通常处理的信号往往具有非零的自相关函数，这使得维纳滤波器的设计变得复杂。来滤波器的拟合时，通过白化处理，我们可以将输入信号转换为白噪声信号，从而简化维纳滤波器的设计和实现。

而在数据生成时，我们将随机初始化白噪声信号，并通过学习的反因果参数来生成具有特定统计特性的时间序列数据。通过这种方式，我们可以保证生成的时间序列与输入的拟合序列具有相似的自相关和功率谱密度。

假设我们输入的信号为$x(n)$，其自相关矩阵为$R_{x x}$，我们将设计一个白化滤波器$\left [ h \right ]$使得输出信号$y(n) \sim \mathcal{N}(0, \sigma^2)$为特定方差的白噪声。基于(9)中的维纳-霍夫方程有:
$$
\begin{bmatrix}r_{x x} (0) & r_{x x} (1) & \cdots & r_{x x} (N) \\ r_{x x} (1) & r_{x x} (0) & \cdots & r_{x x} (N - 1) \\ \vdots & \vdots & \ddots & \vdots \\ r_{x x} (N) & r_{x x} (N - 1) & \cdots & r_{x x} (0) \end{bmatrix} \cdot \begin{bmatrix} h(0) \\ h(1) \\ \vdots \\ h(N) \end{bmatrix} = \begin{bmatrix} \sigma^2 \\ 0 \\ \vdots \\ 0 \end{bmatrix}. \quad (12)
$$
即输入信号的自相关矩阵与白化滤波器的参数相乘得到白噪声的自相关。

准确来说，式(12)相比于维纳滤波，应该更类似于AR模型的参数化谱估计。我们需要求解的参数为$\left [ h \right ]$以及白噪声的方差$\sigma^2$。

通过$\sigma^2$我们可以重新初始化白噪声，并通过反因果的白化滤波器$1 / \left [ h \right ]$来生成时间序列。

本notebook进一步详细演示了上述的过程。


In [ ]:
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))

from s2generator.simulator import WienerFilterSimulator
from s2generator.utils import yule_walker, plot_simulator_statistics